# LLM Fine-Tuning Deep Dive, Part 3 of 3: Comparison & Decision (TensorFlow/Keras)

> **This is Part 3 of a three-notebook fine-tuning arc:**
>
> 1. [Part 1: Data-based techniques](01-llm-finetuning-data-techniques.ipynb) — continued pretraining (full FT), instruction tuning (layer freezing), preference-signal FT.
> 2. [Part 2: Parameter-based techniques + TF quantization](02-llm-finetuning-parameter-techniques.ipynb) — full fine-tuning, partial freezing, TF-native parameter-efficient FT, TFLite quantization.
> 3. **Part 3 (this notebook): Comparison & decision** — head-to-head evaluation of all trained checkpoints, held-out perplexity, ablation study, and the final call on what Riverside House actually deploys.
>
> **Recap:** Parts 1 and 2 trained and saved five checkpoints to `./checkpoints/` on disk. This notebook reloads all five from disk — fresh Python objects, not the same in-memory instances Parts 1-2 trained — and puts them head-to-head.

## Table of Contents (Part 3)

1. [Setup: Reloading All Five Trained Checkpoints](#setup)
2. [Comparing All Five Techniques](#comparing-all-five-techniques)
3. [Deep Dive: Token Probability Analysis](#deep-dive-token-probability-analysis)
4. [Held-Out Perplexity: The Number Riverside Actually Needs](#held-out-perplexity)
5. [Technique Combination Grid: Data × Parameter](#technique-combination-grid)
6. [Ablation Study: What Happens If You Skip a Stage?](#ablation-study)
7. [The Decision: What Do We Actually Hand to Riverside House?](#the-decision)
8. [Further Reading & Scaling Up](#further-reading)

---

## Choose the Training Path

Before comparing numbers, the framing: these six checkpoints aren't "better vs. worse" — they answer *different questions*. The right one to deploy depends on which gap Riverside is trying to close today.

| Model | Trains what | Best for |
| --- | --- | --- |
| Baseline (no FT) | — | Measuring the gap |
| `non-instruction-full` | Domain knowledge (full FT) | Lore-aware text continuation |
| `instruction-ft` | Instruction-following (layer freeze) | Answering specific questions |
| `preference-ft` | Editorial preference (contrastive FT) | House-style writing |
| `partial-freeze` | Domain knowledge (25% of params) | Cost-efficient lore absorption |
| `peft-tf` | Domain knowledge (4% of params) | Cheapest lore absorption |

The comparison below scores *all* on the same held-out prose prediction task — which structurally favours continued-pretraining models. The qualitative reads and the ablation study complete the picture.

In [ ]:
# Re-establishing the full foundation for Part 3
import subprocess, sys, warnings, os, math
required = [
    ('numpy', 'numpy'), ('matplotlib', 'matplotlib'),
    ('tensorflow', 'tensorflow'), ('transformers', 'transformers'),
]
for imp, pkg in required:
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tensorflow as tf
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
sns.set_theme(style='whitegrid', palette='muted')

MODEL_NAME = 'gpt2-medium'
PROMPT = 'Aria Voss stared at the signal counting itself out in prime numbers and'
INSTRUCTION_PREFIX = 'Continue the fiction narrative in the same style:\n\n'

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)


def generate(model, prompt, max_new_tokens=60):
    input_ids = tokenizer(prompt, return_tensors='tf')['input_ids']
    prompt_len = input_ids.shape[1]
    output = model.generate(
        input_ids, max_new_tokens=max_new_tokens,
        do_sample=True, top_p=0.9, temperature=0.8,
        pad_token_id=tokenizer.pad_token_id,
    )
    decoded = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()
    return decoded if decoded else '[model stopped — EOS as first token]'


print(f'Setup complete. Baseline: {generate(base_model, PROMPT, max_new_tokens=30)}')

In [ ]:
# Corpus loader for held-out evaluation
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / 'content'
if not CONTENT_DIR.exists():
    _fallback = Path.cwd() / 'learning' / 'genai' / '04-llm' / 'content'
    if _fallback.exists():
        CONTENT_DIR = _fallback

NOVELS = {
    'scifi': 'the-weight-of-distant-light',
    'fantasy': 'the-tidebound-accord',
    'mystery': 'the-cartographers-cipher',
    'historical': 'the-silk-merchants-daughter',
    'cyberpunk': 'neural-drift',
    'horror': 'the-hollow-beneath',
    'literary': 'the-weight-of-tides',
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob('chapter-*.txt'))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding='utf-8')
            for para in text.split('\n\n'):
                para = para.strip().replace('\n', ' ')
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


print(f'Content directory: {CONTENT_DIR.absolute()}')

### Reloading the Five Fine-Tuned Checkpoints

Each checkpoint is a full `TFGPT2LMHeadModel` saved with `save_pretrained`. `from_pretrained` loads them fresh — no memory sharing between notebooks. `partial-freeze` and `peft-tf` are plain fine-tuned models (no adapter layer to reattach), so reloading them is a single `from_pretrained` call per model.

In [ ]:
# Reload all five fine-tuned checkpoints
checkpoint_paths = {
    'Full fine-tuning': './checkpoints/non-instruction-full',
    'Instruction-tuned (layer freeze)': './checkpoints/instruction-ft',
    'Preference-aligned': './checkpoints/preference-ft',
    'Partial freezing': './checkpoints/partial-freeze',
    'TF-native PEFT': './checkpoints/peft-tf',
}

models = {'Baseline (no FT)': base_model}
missing = []
for name, path in checkpoint_paths.items():
    if os.path.isdir(path) and len(os.listdir(path)) > 0:
        models[name] = TFGPT2LMHeadModel.from_pretrained(path)
        print(f'  loaded  {name:<35} from {path}')
    else:
        missing.append((name, path))
        print(f'  MISSING {name:<35}  ({path})')

if missing:
    print(f'\nWARNING: {len(missing)} checkpoint(s) missing. Run Parts 1 and 2 first.')
else:
    print(f'\nAll {len(models)} models loaded (baseline + 5 fine-tuned).')

---

## Comparing All Five Techniques

Let's see what each model actually produces for the same prompt. Two prompts:
- Plain continuation prompt (no instruction prefix) — fair to `non-instruction-full`, `partial-freeze`, `peft-tf`
- Instruction-prefixed prompt — fair to `instruction-ft`, `preference-ft`

In [ ]:
instruct_prompt = INSTRUCTION_PREFIX + PROMPT + '\n\n'
print(f'Plain prompt      : {PROMPT!r}')
print(f'Instruction prompt: (prefix) + {PROMPT[:60]!r}...')
print()

for name, model in models.items():
    if 'nstruction' in name or 'reference' in name:
        output = generate(model, instruct_prompt, max_new_tokens=50)
        print(f'[{name}] (+ instruction prefix)')
    else:
        output = generate(model, PROMPT, max_new_tokens=50)
        print(f'[{name}]')
    print(f'  "{output}"')
    print()

---

## Deep Dive: What Actually Changed? Token Probability Analysis

Generated text shows *what* the model says, but not *how much* the internal probability distribution shifted. Let's look at exactly how the model's next-token prediction changed after fine-tuning.

**Question:** After fine-tuning on Riverside's catalog, how much more likely is the model to predict domain-specific vocabulary vs. generic English words?

In [ ]:
# Token probability analysis: before vs. after fine-tuning
prompt_for_analysis = "Aria Voss checked the Meridian's Promise and"

domain_words = ['saw', 'discovered', 'noted', 'realized', 'found', 'detected', 'confirmed']
generic_words = ['the', 'a', 'then', 'he', 'she', 'was', 'said']


def get_next_token_probs(model, prompt, candidate_words):
    """Return {word: P(word is next token | prompt)} for each candidate word."""
    input_ids = tokenizer(prompt, return_tensors='tf')['input_ids']
    outputs = model(input_ids, training=False)
    logits = outputs.logits[0, -1, :]  # next-token logits
    probs = tf.nn.softmax(logits).numpy()
    results = {}
    for word in candidate_words:
        word_ids = tokenizer.encode(' ' + word, add_special_tokens=False)
        if word_ids:
            results[word] = float(probs[word_ids[0]])
    return results


# Use the full fine-tuning checkpoint as the "fine-tuned" model for comparison
ft_model = models.get('Full fine-tuning', base_model)

base_domain = get_next_token_probs(base_model, prompt_for_analysis, domain_words)
ft_domain = get_next_token_probs(ft_model, prompt_for_analysis, domain_words)
base_generic = get_next_token_probs(base_model, prompt_for_analysis, generic_words)
ft_generic = get_next_token_probs(ft_model, prompt_for_analysis, generic_words)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Token Probability Shift After Fine-Tuning\n"{prompt_for_analysis}"', fontsize=11, fontweight='bold')
x_d = np.arange(len(domain_words)); x_g = np.arange(len(generic_words)); w = 0.35

ax1.bar(x_d - w/2, [base_domain[wd]*100 for wd in domain_words], w, label='Baseline', alpha=0.8, color='steelblue')
ax1.bar(x_d + w/2, [ft_domain[wd]*100 for wd in domain_words], w, label='Fine-tuned (full FT)', alpha=0.8, color='coral')
ax1.set_xticks(x_d); ax1.set_xticklabels(domain_words, rotation=30, ha='right')
ax1.set_ylabel('Probability (%)'); ax1.set_title('Domain Words'); ax1.legend(); ax1.grid(alpha=0.3, axis='y')
for i, wd in enumerate(domain_words):
    delta = (ft_domain[wd] - base_domain[wd]) * 100
    if abs(delta) > 0.01:
        ax1.text(i + w/2, ft_domain[wd]*100 + 0.001, f'{delta:+.2f}%', ha='center', va='bottom', fontsize=7,
                 color='green' if delta > 0 else 'red', fontweight='bold')

ax2.bar(x_g - w/2, [base_generic[wd]*100 for wd in generic_words], w, label='Baseline', alpha=0.8, color='steelblue')
ax2.bar(x_g + w/2, [ft_generic[wd]*100 for wd in generic_words], w, label='Fine-tuned (full FT)', alpha=0.8, color='lightgreen')
ax2.set_xticks(x_g); ax2.set_xticklabels(generic_words, rotation=30, ha='right')
ax2.set_ylabel('Probability (%)'); ax2.set_title('Generic Words (expected to stay stable)'); ax2.legend(); ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary
print(f'\n{"=" * 70}')
print('Token probability shift summary:')
domain_gains = {wd: (ft_domain[wd] - base_domain[wd]) * 100 for wd in domain_words}
top_3 = sorted(domain_gains.items(), key=lambda kv: kv[1], reverse=True)[:3]
for wd, delta in top_3:
    print(f'  {wd}: {base_domain[wd]*100:.3f}% -> {ft_domain[wd]*100:.3f}% ({delta:+.3f}%)')
n_gained = sum(1 for v in domain_gains.values() if v > 0)
print(f'\n{n_gained}/{len(domain_words)} domain words gained probability after fine-tuning.')
print(f'Full DPO analysis (PyTorch) includes reference model KL divergence — not computed in TF version.')

---

## Held-Out Perplexity: The Number Riverside Actually Needs

Every comparison so far has been qualitative — read a paragraph, judge it by eye. That's not enough to convince an IT lead to deploy a model company-wide. Let's score **all five checkpoints** on the same held-out paragraphs, taken from chapters deep in each novel that **none of the training runs above ever saw** (every training cell used only the first few chapters per novel; this held-out set starts at chapter 11).

**Lower perplexity = higher probability assigned to Riverside's actual prose** = better fit to the manuscript domain.

**Honest caveat:** The instruction-tuned and preference-aligned models were trained to expect the `INSTRUCTION_PREFIX` format. Scoring them here *without* that prefix answers: "how good is this model as a general house-style language model?" — which is what the knowledge-base use case actually needs. It's fair to expect instruction/preference models to score worse on plain prose perplexity even if they're better at the tasks they were tuned for.

In [ ]:
def load_holdout_paragraphs(start_chapter_idx=10, chapters_per_novel=2, min_len=200):
    """Load paragraphs from later chapters — guaranteed unseen by every training run."""
    paragraphs = []
    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob('chapter-*.txt'))
        for path in chapter_files[start_chapter_idx : start_chapter_idx + chapters_per_novel]:
            text = path.read_text(encoding='utf-8')
            for para in text.split('\n\n'):
                para = para.strip().replace('\n', ' ')
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


holdout_paragraphs = load_holdout_paragraphs()
print(f'Held-out set: {len(holdout_paragraphs)} paragraphs from chapter 11+ of each novel')


def compute_holdout_loss(model, paragraphs, max_length=128):
    """Average cross-entropy loss on a set of held-out paragraphs."""
    losses = []
    for para in paragraphs:
        enc = tokenizer(para, truncation=True, max_length=max_length, return_tensors='tf')
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        outputs = model(input_ids, attention_mask=attention_mask, training=False)
        logits = outputs.logits[:, :-1, :]
        labels = input_ids[:, 1:]
        pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)
        token_loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        mean_loss = tf.reduce_sum(token_loss * pad_mask) / (tf.reduce_sum(pad_mask) + 1e-9)
        losses.append(float(mean_loss))
    return sum(losses) / len(losses)


print(f'\n{"=" * 70}\nHeld-out evaluation (lower is better):\n{"=" * 70}')
holdout_results = {}
for name, model in models.items():
    avg_loss = compute_holdout_loss(model, holdout_paragraphs)
    perplexity = math.exp(avg_loss)
    holdout_results[name] = {'loss': avg_loss, 'perplexity': perplexity}
    print(f'  {name:<38} loss={avg_loss:6.3f}   perplexity={perplexity:8.1f}')
print(f'{"=" * 70}')

In [ ]:
# Visualize held-out perplexity
ranked = sorted(holdout_results.items(), key=lambda kv: kv[1]['perplexity'])
names_r = [n for n, _ in ranked]
ppls_r = [res['perplexity'] for _, res in ranked]
bar_colors_r = ['#2a9d8f' if i == 0 else '#457b9d' for i in range(len(ranked))]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names_r[::-1], ppls_r[::-1], color=bar_colors_r[::-1], edgecolor='black', alpha=0.85)
ax.set_xlabel('Held-out perplexity (lower = better fit to Riverside prose)', fontsize=10)
ax.set_title('Held-Out Perplexity Across All Checkpoints\n(unseen chapters — the closest thing to a deployment test)', fontsize=11, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
for bar, ppl in zip(bars[::-1], ppls_r):
    ax.text(ppl + 0.5, bar.get_y() + bar.get_height() / 2, f'{ppl:.1f}', va='center', fontsize=9)
ax.legend(
    handles=[mpatches.Patch(facecolor='#2a9d8f', label='Best perplexity'),
             mpatches.Patch(facecolor='#457b9d', label='Other checkpoints')],
    loc='lower right', fontsize=8
)
plt.tight_layout()
plt.show()

print(f'\nBest held-out fit: "{ranked[0][0]}" (perplexity={ranked[0][1]["perplexity"]:.1f})')
print('Reminder: this metric rewards next-token prediction on plain prose.')
print('Instruction/preference models optimised for a different task — their lower prose perplexity')
print('is expected, not a failure.')

---

## Technique Combination Grid: Data × Parameter

The two fine-tuning axes are **independent choices** — any data objective can be paired with any parameter strategy. This notebook trained five checkpoints covering four of the nine cells:

| | **Full FT (100%)** | **Partial Freeze (~21%)** | **TF-native PEFT (~4-5%)** |
| --- | --- | --- | --- |
| **Continued pretraining** | `non-instruction-full` | `partial-freeze` | `peft-tf` |
| **Instruction tuning (SFT)** | not trained | not trained | `instruction-ft` (via layer freeze) |
| **Preference alignment** | not trained | not trained | `preference-ft` (contrastive loss) |

The cell below builds this 3×3 heatmap with real held-out perplexity numbers for the trained cells.

In [ ]:
# 3x3 technique combination grid
data_objectives = ['Continued\nPretraining', 'Instruction\nTuning (SFT)', 'Preference\nAlignment']
param_strategies = ['Full FT\n(100%)', 'Partial Freeze\n(~21%)', 'TF-PEFT\n(<5%)']

# Mapping (row, col) -> checkpoint name in holdout_results
checkpoint_map = {
    (0, 0): 'Full fine-tuning',
    (0, 1): 'Partial freezing',
    (0, 2): 'TF-native PEFT',
    (1, 2): 'Instruction-tuned (layer freeze)',
    (2, 2): 'Preference-aligned',
}

# Build the heatmap grids
ppl_grid = np.full((3, 3), np.nan)
trained_mask = np.zeros((3, 3), dtype=bool)

for (r, c), ckpt_name in checkpoint_map.items():
    if ckpt_name in holdout_results:
        ppl_grid[r, c] = holdout_results[ckpt_name]['perplexity']
        trained_mask[r, c] = True

# Parameter percentages (from Part 2's real numbers — approximate)
param_pcts_row = [100.0, 21.0, 4.5]
param_grid = np.array([param_pcts_row] * 3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Data × Parameter Technique Grid — Trained Combinations\n(grey = not trained in this run)', fontsize=12, fontweight='bold')


def draw_grid(ax, values, fmt, title, cmap_name, label):
    display = np.where(trained_mask, values, np.nan)
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color='#cccccc')
    valid = display[trained_mask]
    vmin, vmax = (float(valid.min()), float(valid.max())) if valid.size > 1 else (0.0, 1.0)
    im = ax.imshow(display, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    for r in range(3):
        for c in range(3):
            if trained_mask[r, c]:
                txt = fmt.format(display[r, c])
                ax.text(c, r, txt, ha='center', va='center', fontsize=11, fontweight='bold',
                        color='white' if display[r, c] < (vmin + (vmax - vmin) * 0.6) else 'black')
            else:
                ax.text(c, r, '—\n(not trained)', ha='center', va='center', fontsize=9, color='#555555', style='italic')
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(param_strategies, fontsize=9)
    ax.set_yticklabels(data_objectives, fontsize=9)
    ax.set_xlabel('Parameter strategy', fontsize=10, fontweight='bold')
    ax.set_ylabel('Data objective', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label=label)


draw_grid(ax1, ppl_grid, '{:.1f}', 'Held-Out Perplexity\n(lower = better)', 'YlOrRd_r', 'Perplexity')
draw_grid(ax2, param_grid, '{:.1f}%', 'Trainable Parameters %\n(lower = cheaper)', 'Blues_r', 'Trainable %')
plt.tight_layout()
plt.show()

print(f'\n{"=" * 70}')
print('Grid Takeaways:')
print(f'{"=" * 70}')
trained_cells = [(r, c, ppl_grid[r, c]) for (r, c) in checkpoint_map if trained_mask[r, c]]
trained_cells.sort(key=lambda x: x[2])
best_r, best_c, best_ppl = trained_cells[0]
print(f'  Best perplexity: [{data_objectives[best_r].replace(chr(10), " ")}] x [{param_strategies[best_c].replace(chr(10), " ")}] -> {best_ppl:.1f}')
print('  Row 0 (continued pretraining): perplexity rises as trainable params shrink — Full FT > Partial > PEFT.')
print('  Rows 1-2 (instruction/preference): higher perplexity on plain prose — expected (trained for different task).')

---

## Ablation Study: What Happens If You Skip a Stage?

**Riverside's question for this section:** the launch date got moved up. If we have to cut a corner to ship on time, which corner is safe to cut, and which one breaks the assistant?

### Experiment 1: Skip Continued Pretraining (Base → Instruction Tuning Directly)

**Setup:** Train instruction tuning on a base model that has never seen the domain corpus.
**Expected result:** Model learns the instruction format but doesn't know domain vocabulary.

**Example — without domain pretraining:**
- "Continue: Aria Voss checked the Meridian and..." → generic space opera tropes, no story-specific terminology

**Verdict:** You **can** skip continued pretraining if your domain vocabulary overlaps heavily with general English (e.g., customer support chatbot). You **cannot** skip it for Riverside — the catalog is full of invented terminology (`Meridian's Promise`, `Tidebound Accord`, `void stone`) that the base model has never seen.

---

### Experiment 2: Skip Instruction Tuning (Continued Pretraining → Preference FT Directly)

**Setup:** Run preference-signal FT on a model that only has domain knowledge but no instruction-following.
**Expected result:** The preference pairs are sequences of prose, not (instruction, completion) pairs — so the model learns to slightly prefer certain prose styles but not to *follow* an instruction.

**Verdict:** Preference fine-tuning **depends on** instruction tuning. The contrastive loss (prefer chosen over rejected) assumes the model already knows how to follow the `INSTRUCTION_PREFIX` format; without instruction tuning, the chosen/rejected distinction is too noisy.

---

### Experiment 3: Wrong Order (Preference FT → Instruction Tuning)

**Setup:** Run preference-signal FT first, then instruction tuning afterward.
**Expected result:** The instruction tuning updates overwrite the preference signal (instruction tuning's LR and data volume dominate the preference fine-tuning).

**Verdict:** Always do preference fine-tuning **last**. The correct order every time Riverside re-trains: continued pretraining → instruction tuning → preference FT.

In [ ]:
# Ablation: demonstrate what instruction tuning without domain pretraining looks like
# (qualitative only — we don't run a full new training loop here)
ablation_prompts = [
    ('Domain knowledge', PROMPT, False),
    ('Instruction following', INSTRUCTION_PREFIX + 'Who is Aria Voss?' + '\n\n', True),
    ('Preference (style)', INSTRUCTION_PREFIX + 'Aria checked the Meridian and' + '\n\n', True),
]

print('Ablation: what each checkpoint can and cannot do')
print('=' * 70)
for prompt_type, prompt, use_instruction in ablation_prompts:
    print(f'\n[{prompt_type}]')
    for name, model in [('Baseline', base_model), ('Full FT', models.get('Full fine-tuning', base_model)),
                        ('Instruction-ft', models.get('Instruction-tuned (layer freeze)', base_model)),
                        ('Preference-ft', models.get('Preference-aligned', base_model))]:
        output = generate(model, prompt, max_new_tokens=35)
        flag = '' if (('nstruction' in name or 'reference' in name) == use_instruction) else ' [MISMATCHED FORMAT]'
        print(f'  {name:<16}: "{output}"{flag}')

---

## The Decision: What Do We Actually Hand to Riverside House?

Having looked at qualitative comparisons, token probability shifts, held-out perplexity, the combination grid, and the ablation study — what's the final call?

### Scoring the Candidates

,
,
,
,
,
,
5
,
,
,
,
,
,
5
50
,
,
2
,
,
]},{
:
,
:null,
:
,
:{},
:[],
:[
,
,
,
,
,

In [ ]:
# Final decision summary: score each model on a composite rubric
print('=' * 70)
print('FINAL DEPLOYMENT RECOMMENDATION FOR RIVERSIDE HOUSE')
print('=' * 70)

decision_models = [
    ('non-instruction-full', 'Lore knowledge (full FT)', 'Best prose perplexity; no instruction-following'),
    ('instruction-ft', 'Editing assistant', 'Instruction-following + lore; recommended for knowledge base'),
    ('preference-ft', 'House-style editing assistant', 'Best assistant quality; deploy for ghostwriters'),
    ('partial-freeze', 'Budget lore option', 'Good perplexity at lower cost; not for assistant'),
    ('peft-tf', 'Cheapest lore option', 'Lowest cost; lowest quality ceiling'),
]

ckpt_name_map = {
    'non-instruction-full': 'Full fine-tuning',
    'instruction-ft': 'Instruction-tuned (layer freeze)',
    'preference-ft': 'Preference-aligned',
    'partial-freeze': 'Partial freezing',
    'peft-tf': 'TF-native PEFT',
}
for ckpt_name, role, rationale in decision_models:
    ppl_key = ckpt_name_map.get(ckpt_name)
    ppl = holdout_results.get(ppl_key, {}).get('perplexity', float('nan'))
    print(ckpt_name.ljust(28) + '  ppl=' + str(round(ppl, 1)).rjust(7) + '  role: ' + role)
    print('    ' + rationale)
    print()

print('RECOMMENDATION:')
print("  Editing assistant (ghostwriters):    deploy ./checkpoints/preference-ft")
print("  Knowledge base (factual Q&A):        deploy ./checkpoints/instruction-ft")
print("  Single-model deployment:             ./checkpoints/preference-ft (subsumes instruction-ft)")


In [ ]:
import os

print('Final checkpoint inventory:')
print('=' * 60)
all_ckpts = [
    ('./checkpoints/non-instruction-full', 'continued pretraining / full FT (Part 1)'),
    ('./checkpoints/instruction-ft', 'instruction tuning / layer freeze (Part 1)'),
    ('./checkpoints/preference-ft', 'preference-signal FT / contrastive loss (Part 1)'),
    ('./checkpoints/partial-freeze', 'partial freezing / 25% trainable (Part 2)'),
    ('./checkpoints/peft-tf', 'TF-native PEFT / 4-5% trainable (Part 2)'),
]
for path, desc in all_ckpts:
    exists = os.path.isdir(path)
    n_files = len(os.listdir(path)) if exists else 0
    status = 'OK     ' if (exists and n_files > 0) else 'MISSING'
    print(status + '  ' + path.ljust(45) + ' (' + desc + ')')
print('=' * 60)

print()
print('Held-out perplexity summary:')
print('  ' + 'Checkpoint'.ljust(38) + ' Perplexity')
print('  ' + '=' * 52)
for name, res in sorted(holdout_results.items(), key=lambda kv: kv[1]['perplexity']):
    ppl_str = str(round(res['perplexity'], 1)).rjust(8)
    print('  ' + name.ljust(38) + ppl_str)


---

## What This Notebook Covered (and What It Didn't)

### Covered

| Concept | Status |
| --- | --- |
| Continued pretraining (full FT) | Trained, evaluated |
| Instruction tuning (layer freezing, TF substitute for LoRA) | Trained, evaluated |
| Preference-signal FT (contrastive loss, TF substitute for DPO) | Trained, evaluated |
| Partial freezing (25% trainable) | Trained, evaluated |
| TF-native PEFT (4-5% trainable, ultra-partial freeze) | Trained, evaluated |
| Held-out perplexity across all checkpoints | Computed |
| Data × parameter combination grid | Visualized |
| Ablation study (skip a stage, wrong order) | Qualitatively demonstrated |

### Not Covered (PyTorch-only, or GPU-only)

| Concept | Why not covered | Where to find it |
| --- | --- | --- |
| LoRA/PEFT (<1% trainable adapters) | PyTorch-only | PyTorch version of this series |
| QLoRA (4-bit quantized base + LoRA) | GPU + bitsandbytes required | PyTorch version |
| Full DPO with reference model | TRL is PyTorch-only | PyTorch version |
| PPO-based RLHF | Requires reward model + complex policy optimization | Research-level infrastructure |

---

## Further Reading & Scaling Up

| Topic | Resource |
| --- | --- |
| DPO paper | Rafailov et al. (2023), "Direct Preference Optimization" |
| LoRA paper | Hu et al. (2021), "LoRA: Low-Rank Adaptation of Large Language Models" |
| QLoRA paper | Dettmers et al. (2023), "QLoRA: Efficient Finetuning of Quantized LLMs" |
| TRL library (PyTorch) | huggingface.co/docs/trl |
| PEFT library (PyTorch) | huggingface.co/docs/peft |
| Keras NLP | keras.io/keras_nlp — community TF/Keras NLP models |
| TFLite quantization | tensorflow.org/lite/performance/post_training_quantization |

**Scaling up from gpt2-medium:** The same three techniques (continued pretraining, instruction tuning, preference FT) apply to any decoder-only model. For Llama-3, Mistral, or larger GPT variants:
- Use the PyTorch version of this series (LoRA/PEFT handles adapter injection for you)
- For TF: consider `keras-nlp`'s fine-tuning utilities for Llama and Gemma
- For QLoRA: you need a GPU with ≥8 GB VRAM for 7B models


---

## Series Summary: The Full Fine-Tuning Arc

Three notebooks, five trained checkpoints, one deployment decision:

| Part | What it covered | Key TF/PyTorch difference |
| --- | --- | --- |
| Part 1 | Data-based techniques: continued pretraining, instruction tuning, preference FT | Layer freezing (not LoRA); contrastive loss (not DPO) |
| Part 2 | Parameter-based: full FT, partial freeze, TF-native PEFT; TFLite quantization | Ultra-partial freeze (not LoRA); TFLite (not `quantize_dynamic`) |
| Part 3 | Held-out comparison, combination grid, ablation, final decision | Same analysis; slightly fewer trained cells than PyTorch version |

**The two axes, one more time:**

- **Data axis** (what to learn): continued pretraining → instruction tuning → preference FT. The order matters and builds on itself.
- **Parameter axis** (how much to move): full fine-tuning → partial freezing → parameter-efficient. A smaller budget costs quality; the right choice depends on Riverside's hardware and how often they retrain.

**For Riverside House's one-laptop constraint:** `instruction-ft` (or `preference-ft` for the editing assistant) is the practical recommendation — a meaningful improvement over the base model at a cost that fits in available CPU RAM.
